# Third Experiment 

In [ ]:
import bayesian_optimization as bo
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel
from scipy.special import comb
from scipy.stats import norm
import seaborn as sns
from sklearn.gaussian_process.kernels import (
    ConstantKernel as C,
    Matern,
    WhiteKernel,
)
sns.set()

## Experiment 3


- Note starting point matter, if the starting points are closer to the true optimal, very small alphas would win.
- Noise also matters, when std_noise =0.2, EI outperforms linear increase/decrease in alpha. When std_noice =0.3. Linear increase alpha works well
- However, if starting point is very far away from the true optimal, large alpha will win

In [ ]:
# set up true objective
import numpy as np
import warnings
warnings.filterwarnings('ignore')

def f1(x):
    """
    Drop-Wave function on [0, 1]², vectorised.

    Input
    -----
    x : array_like
        Shape (2,)  – a single point
        Shape (N,2) – N points
        Each coordinate must lie in [0, 1].

    Output
    ------
    float or ndarray
        Scalar if the input is shape (2,); 1-D array of length N if (N,2).
    """
    x = np.asarray(x, dtype=float)

    # --- rescale from [0,1] to the classical domain [-5.12, 5.12] ---
    u0 = 10.24 * x[..., 0] - 5.12
    u1 = 10.24 * x[..., 1] - 5.12

    # --- Drop-Wave formula ---
    r2 = u0**2 + u1**2               # squared radius
    r  = np.sqrt(r2)                 # radius
    value = -(1 + np.cos(2 * r)) / (0.5 * r2 + 2)

    return -2*value

bounds = np.asarray(((0, 1),(0,1)))
print(bounds)
#x_star = np.linspace(*bounds, 10000)
n_points = 100
# Create a vector for each coordinate axis
x1s = np.linspace(bounds[0][0], bounds[0][1], n_points)
x2s = np.linspace(bounds[1][0], bounds[1][1], n_points)
# Form a mesh grid, then flatten to (N, 2)
X1, X2 = np.meshgrid(x1s, x2s)                 # each is (n_points, n_points)
x_star = np.column_stack([X1.ravel(), X2.ravel()])

print(x_star)
f_star = f1(x_star)
true_max = np.max(f_star)
true_max_location = x_star[np.argmax(f_star)]

plt.figure(figsize=(7,5))
#plt.plot(x_star, f_star)
plt.scatter(x_star[:,0],x_star[:,1],c=f_star)

In [ ]:
# define kenel
dim = 2
kern_signal = C(1.0, (1e-3, 1e3)) * Matern(
            length_scale=np.ones(dim),
            length_scale_bounds=(1e-3, 1e3),
            nu=2.5,
        )
kern_noise = WhiteKernel(
        noise_level=1e-6,
        noise_level_bounds=(1e-3, 1),
    )
kernel=kern_signal + kern_noise

# create starting data
std_noise = 0.01
np.random.seed(42)
n_init = 5
x_init = np.random.uniform(low=bounds[:,0], high=bounds[:,1], size=(n_init,dim))
#x_init = np.linspace(bounds[0], bounds[1],n_init)
#x_init = np.linspace(bounds[0], 1.25, 5)
print(x_init)
y_init = f1(x_init) + np.random.normal(0, std_noise, size=n_init)

# optimizer parameter -linear increase
n_iter = 25
g = 1
alpha_low = 0.1
acq_samples = dim*10
optimizer = bo.BayesianOptimizer(
    func = f1,
    kernel = kernel,
    bounds = np.array(bounds).reshape(dim, 2),
    g= np.ones(n_iter)*g,
    alpha = np.linspace(alpha_low, 1.0, n_iter),
    x_init = x_init.reshape(-1,dim),
    y_init = y_init.reshape(-1,1),
    xi = 0,
    n_iter= n_iter,
    acq_samples=acq_samples, # num starting points to optimize acquization
    random_state=42,

)

# run optimization
opt_result = optimizer.simulate_optimization(noise_std = std_noise, max_val=true_max, visualize=False, verbose= False)
print(opt_result)
linear_alpha_result = {"cumulative_regrets": np.sum(opt_result["inst_regrets"]),  "dataset_x": optimizer.X, "dataset_y": optimizer.Y, **opt_result}
alpha_result = {"linear_increase":linear_alpha_result}

In [ ]:
import warnings
warnings.filterwarnings(action='once')
# optimizer parameter linear decrease
g = 1
optimizer = bo.BayesianOptimizer(
    func = f1,
    kernel = kernel,
    bounds = np.array(bounds).reshape(dim, 2),
    g= np.ones(n_iter)*g,
    alpha = np.linspace(1, alpha_low, n_iter),
    x_init = x_init.reshape(-1,dim),
    y_init = y_init.reshape(-1,1),
    xi = 0,
    n_iter= n_iter,
    acq_samples=acq_samples, # num starting points to optimize acquization
    random_state=42,

)

# run optimization
opt_result = optimizer.simulate_optimization(noise_std = std_noise, max_val=true_max, verbose= False)
print(opt_result)
linear_alpha_result = {"cumulative_regrets": np.sum(opt_result["inst_regrets"]),  "dataset_x": optimizer.X, "dataset_y": optimizer.Y, **opt_result}
alpha_result["linear_decay"] = linear_alpha_result

In [ ]:

# color
def plot_bo_color(true_x, true_y, data_x, data_y, title):
    n_color = data_y.shape[0]
    plt.figure(figsize=(8,4))
    plt.plot(true_x, true_y, 'k-', label='true f(x)')
    sc = plt.scatter(optimizer.X[n_init:], optimizer.Y[n_init:], c=np.arange(n_color), cmap='viridis', s=60, edgecolor='k')
    plt.colorbar(sc, label='iteration index')
    plt.legend()
    plt.xlabel('x')
    plt.ylabel('f(x)')
    plt.title(title)
    plt.show()

In [ ]:
# by number
def plot_bo_number(true_x, true_y, data_x, data_y, title):
    plt.figure(figsize=(8,4))
    plt.plot(true_x, true_y, 'k-', alpha=0.5)
    plt.scatter(data_x, data_y, c='C1', s=50, edgecolor='k')
    for i,(x,y) in enumerate(zip(data_x,data_y)):
        plt.text(x, y, str(i), color='black',
                fontsize=9, ha='center', va='bottom')
    plt.xlabel('x'); plt.ylabel('f(x)')
    plt.title(title)
    plt.show()

In [ ]:
# RUN FOR ALL ALPHAS
all_alphas = np.arange(0.1, 1.1, 0.1)
all_alphas= np.append(all_alphas, 0.01)
for alpha in all_alphas:
    optimizer = bo.BayesianOptimizer(
        func = f1,
        kernel = kernel,
        bounds = np.array(bounds).reshape(dim, 2),
        g= np.ones(n_iter)*g,
        alpha = np.ones(n_iter)* alpha,
        x_init = x_init.reshape(-1,dim),
        y_init = y_init.reshape(-1,1),
        xi = 0,
        n_iter= n_iter,
        acq_samples=acq_samples, # num starting points to optimize acquization
        random_state=42
    )
    opt_result = optimizer.simulate_optimization(noise_std = std_noise, max_val=true_max, verbose= False)
    print(opt_result)
    temp_result = {"cumulative_regrets": np.sum(opt_result["inst_regrets"]),  "dataset_x": optimizer.X, "dataset_y": optimizer.Y, **opt_result}
    alpha_result[alpha] = temp_result
    # visualize
    fig, (ax_f, ax_mu) = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)

    # ---------- left panel: true objective f_star ----------
    sc_f = ax_f.scatter(
            x_star[:, 0], x_star[:, 1],
            c=f_star.ravel(), cmap='viridis', alpha=0.5, label='true f(x)'
    )
    data_x, data_y = optimizer.X[n_init:], optimizer.Y[n_init:]
    ax_f.scatter(
            data_x[:, 0], data_x[:, 1],
            c=data_y.ravel(), cmap='viridis', edgecolor='k', s=50
    )

    for i, (pt, label) in enumerate(zip(data_x, data_y)):
        ax_f.text(pt[0], pt[1], str(i),
                color='black', fontsize=9,
                ha='center', va='bottom')

    ax_f.set_title("True function $f(x)$")
    ax_f.set_xlabel(r"$x_0$")
    ax_f.set_ylabel(r"$x_1$")
    fig.colorbar(sc_f, ax=ax_f, label='f value')

    # ---------- right panel: GP mean μ ----------
    mu, std = optimizer.predict_tempered(x_star, alpha)   # x_star already (N,2)

    sc_mu = ax_mu.scatter(
            x_star[:, 0], x_star[:, 1],
            c=mu.ravel(), cmap='plasma', alpha=0.5, label='GP mean', zorder=-1
    )
    ax_mu.scatter(
            data_x[:, 0], data_x[:, 1],
            c=data_y.ravel(), cmap='plasma', edgecolor='k', s=50
    )

    ax_mu.set_title(r"GP mean $\mu(x)$ (α = {})".format(alpha))
    ax_mu.set_xlabel(r"$x_0$")
    # y-label shared, so only set on the first axis
    fig.colorbar(sc_mu, ax=ax_mu, label='μ value')

    plt.tight_layout()
    plt.show()
    #plt.plot(x_star, mu, 'b--', label='GP mean')
    
    # ±1σ band
    # plt.fill_between(
    #     x_star.ravel(),
    #     (mu - std).ravel(),
    #     (mu + std).ravel(),
    #     color='blue',
    #     alpha=0.2,
    #     label=r'$\pm1\,\sigma$'
    # )
    # plt.legend()
    plt.show()
    #print(STOP)
    #plot_bo_number(x_star, f_star, optimizer.X[n_init:], optimizer.Y[n_init:], title= f"expected_improvement: alpha={alpha}")


In [ ]:
alpha_result.keys()

In [ ]:
alpha_result["linear_decay"].keys()

In [ ]:
cols = ['cumulative_regrets', 'best_iteration_observed', 'best_observed', 'best_observed_x', 'best_iteration_regret']
alpha_result_df = pd.DataFrame({
    key: {c: sub.get(c, None) for c in cols}
    for key, sub in alpha_result.items()
}).T

In [ ]:
alpha_result_df

In [ ]:
# ------------------------------------------------------------------
# MAIN LOOP
# ------------------------------------------------------------------
seed_frames = []                # collect per-seed DataFrames here
seeds         = range(1)
for seed in seeds:
    rng = np.random.default_rng(seed)

    # draw one random starting point and evaluate f1
    x_init = rng.uniform(low=np.array(bounds)[:, 0],
                         high=np.array(bounds)[:, 1],
                         size=(n_init, dim))
    y_init = f1(x_init).reshape(-1, 1)

    alpha_result = {}
    for alpha in all_alphas:
        optimizer = bo.BayesianOptimizer(
            func=f1,
            kernel=kernel,
            bounds=np.asarray(bounds).reshape(dim, 2),
            g=np.full(n_iter, g),
            alpha=np.full(n_iter, alpha),
            x_init=x_init,
            y_init=y_init,
            xi=0,
            n_iter=n_iter,
            acq_samples=acq_samples,
            random_state=seed,         # so GP + acquisition are reproducible
        )
        opt_res = optimizer.simulate_optimization(
            noise_std=std_noise, max_val=true_max, verbose=False
        )

        alpha_result[alpha] = {
            "cumulative_regrets": np.sum(opt_res["inst_regrets"]),
            "best_iteration_observed": opt_res["best_iteration_observed"],
            "best_observed": opt_res["best_observed"],
            "best_observed_x": opt_res["best_observed_x"],
            "best_iteration_regret": opt_res["best_iteration_regret"],
            "dataset_x": optimizer.X,          # keep full traces if you need them
            "dataset_y": optimizer.Y,
        }

    # turn dict → tidy DataFrame, then tag with the current seed
    cols = [
        "cumulative_regrets",
        "best_iteration_observed",
        "best_observed",
        "best_observed_x",
        "best_iteration_regret",
    ]
    alpha_df = (
        pd.DataFrame({k: {c: v.get(c) for c in cols} for k, v in alpha_result.items()})
        .T.reset_index()
        .rename(columns={"index": "alpha"})
        .assign(seed=seed)
    )
    seed_frames.append(alpha_df)

# ------------------------------------------------------------------
# STACK EVERYTHING TOGETHER
# ------------------------------------------------------------------
alpha_results_all = pd.concat(seed_frames, ignore_index=True)

In [ ]:
alpha_results_all.to_csv('third_exp.csv')

In [ ]:
def build_alpha_schedule(kind: str, const_val: float | None = None,
                         n_steps: int = n_iter) -> np.ndarray:
    """
    Returns an array of length `n_steps` describing α_t.
    kind ∈ {"constant", "linear_increase", "linear_decay"}.
    For "constant" you must also give `const_val`.
    """
    if kind == "constant":
        if const_val is None:
            raise ValueError("const_val required for constant schedule")
        return np.full(n_steps, const_val, dtype=float)
    elif kind == "linear_increase":
        return np.linspace(0.01, 1.0, n_steps, dtype=float)
    elif kind == "linear_decay":
        return np.linspace(1.0, 0.01, n_steps, dtype=float)
    else:
        raise ValueError(f"unknown schedule '{kind}'")

# ------------------------------------------------------------------
# MAIN LOOP: sweep seeds × schedules × (constant-α values)
# ------------------------------------------------------------------
records = []          # list of flat dicts → DataFrame later
seeds   = range(10)
for seed in seeds:
    rng = np.random.default_rng(seed)

    # one random initial design per seed
    x_init = rng.uniform(low=np.array(bounds)[:, 0],
                         high=np.array(bounds)[:, 1],
                         size=(n_init, dim))
    y_init = f1(x_init).reshape(-1, 1)

    # -------- 1) constant-α experiments --------
    for alpha_const in all_alphas:
        alpha_vec = build_alpha_schedule("constant", const_val=alpha_const)

        optimizer = bo.BayesianOptimizer(
            func=f1, kernel=kernel,
            bounds=np.asarray(bounds).reshape(dim, 2),
            g=np.full(n_iter, g),
            alpha=alpha_vec,
            x_init=x_init, y_init=y_init,
            xi=0, n_iter=n_iter,
            acq_samples=acq_samples,
            random_state=seed,
        )
        res = optimizer.simulate_optimization(
            noise_std=std_noise, max_val=true_max, verbose=False
        )

        records.append({
            "seed": seed,
            "schedule": "constant",
            "alpha_param": alpha_const,
            "cumulative_regrets": res["inst_regrets"].sum(),
            "best_iteration_observed": res["best_iteration_observed"],
            "best_observed": res["best_observed"],
            "best_observed_x": res["best_observed_x"],
            "best_iteration_regret": res["best_iteration_regret"],
        })

    # -------- 2) linear-increase (0.01 → 1.0) --------
    for sched_name in ["linear_increase", "linear_decay"]:
        alpha_vec = build_alpha_schedule(sched_name)

        optimizer = bo.BayesianOptimizer(
            func=f1, kernel=kernel,
            bounds=np.asarray(bounds).reshape(dim, 2),
            g=np.full(n_iter, g),
            alpha=alpha_vec,
            x_init=x_init, y_init=y_init,
            xi=0, n_iter=n_iter,
            acq_samples=acq_samples,
            random_state=seed,
        )
        res = optimizer.simulate_optimization(
            noise_std=std_noise, max_val=true_max, verbose=False
        )

        records.append({
            "seed": seed,
            "schedule": sched_name,
            "alpha_param": None,            # not applicable for linear schedules
            "cumulative_regrets": res["inst_regrets"].sum(),
            "best_iteration_observed": res["best_iteration_observed"],
            "best_observed": res["best_observed"],
            "best_observed_x": res["best_observed_x"],
            "best_iteration_regret": res["best_iteration_regret"],
        })

# ------------------------------------------------------------------
# FINAL TIDY TABLE
# ------------------------------------------------------------------
alpha_results_all = pd.DataFrame.from_records(records)

In [ ]:
alpha_results_all.to_csv('third_exp1.csv')

In [ ]:
def build_alpha_schedule2(kind: str, const_val: float | None = None,
                         n_steps: int = n_iter) -> np.ndarray:
    """
    Returns an array of length `n_steps` describing α_t.
    kind ∈ {"constant", "linear_increase", "linear_decay"}.
    For "constant" you must also give `const_val`.
    """
    if kind == "constant":
        if const_val is None:
            raise ValueError("const_val required for constant schedule")
        return np.full(n_steps, const_val, dtype=float)
    elif kind == "linear_increase2":
        return np.linspace(0.5, 1.0, n_steps, dtype=float)
    elif kind == "linear_decay2":
        return np.linspace(1.0, 0.5, n_steps, dtype=float)
    else:
        raise ValueError(f"unknown schedule '{kind}'")
    
records = []          # list of flat dicts → DataFrame later
seeds   = range(10)
for seed in seeds:
    rng = np.random.default_rng(seed)

    # one random initial design per seed
    x_init = rng.uniform(low=np.array(bounds)[:, 0],
                         high=np.array(bounds)[:, 1],
                         size=(n_init, dim))
    y_init = f1(x_init).reshape(-1, 1)
    for sched_name in ["linear_increase2", "linear_decay2"]:
        alpha_vec = build_alpha_schedule2(sched_name)

        optimizer = bo.BayesianOptimizer(
            func=f1, kernel=kernel,
            bounds=np.asarray(bounds).reshape(dim, 2),
            g=np.full(n_iter, g),
            alpha=alpha_vec,
            x_init=x_init, y_init=y_init,
            xi=0, n_iter=n_iter,
            acq_samples=acq_samples,
            random_state=seed,
        )
        res = optimizer.simulate_optimization(
            noise_std=std_noise, max_val=true_max, verbose=False
        )

        records.append({
            "seed": seed,
            "schedule": sched_name,
            "alpha_param": None,            # not applicable for linear schedules
            "cumulative_regrets": res["inst_regrets"].sum(),
            "best_iteration_observed": res["best_iteration_observed"],
            "best_observed": res["best_observed"],
            "best_observed_x": res["best_observed_x"],
            "best_iteration_regret": res["best_iteration_regret"],
        })
alpha_results_all2 = pd.DataFrame.from_records(records)

In [ ]:
alpha_results_all2.to_csv('third_exp2.csv')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# 1 · Load the results
# ------------------------------------------------------------------
df = pd.read_csv("third_exp1.csv")

# ------------------------------------------------------------------
# 2 · Build a pretty x-axis label for each run
# ------------------------------------------------------------------
def make_label(row):
    """Return 'α=0.30' for constant schedules, else 'linear_increase' …"""
    if row["schedule"] == "constant":
        return f"α={row['alpha_param']:.2g}"
    return row["schedule"]

df["label"] = df.apply(make_label, axis=1)

# Desired left-to-right order: all constant-α ascending, then special schedules
alpha_vals   = sorted(df.loc[df["schedule"] == "constant", "alpha_param"].unique())
cat_order    = [f"α={a:.2g}" for a in alpha_vals]                       # e.g. α=0.01 … α=1
cat_order   += [s for s in ["linear_increase", "linear_decay",
                            "linear_increase2", "linear_decay2"]
                if s in df["schedule"].unique()]
df["label"]  = pd.Categorical(df["label"], categories=cat_order, ordered=True)

# Map category → numeric x-coordinate
x_lookup = {lab: i for i, lab in enumerate(cat_order)}
df["x"]  = df["label"].map(x_lookup)

# ------------------------------------------------------------------
# 3 · Plot: two vertically stacked scatter panels
# ------------------------------------------------------------------
fig, (ax_reg, ax_best) = plt.subplots(
    nrows=2, ncols=1, figsize=(12, 8), sharex=True
)

# --- panel 1: cumulative regret ---
ax_reg.scatter(df["x"], df["cumulative_regrets"], s=40, alpha=0.8)
ax_reg.set_ylabel("Cumulative regret")

# --- panel 2: best observed value ---
ax_best.scatter(df["x"], df["best_observed"],  s=40, alpha=0.8, color="tab:orange")
ax_best.set_ylabel("Best observed value")

# --- common x-axis cosmetics ---
ax_best.set_xticks(range(len(cat_order)))
ax_best.set_xticklabels(cat_order, rotation=45, ha="right")
ax_best.set_xlabel("Schedule / constant α")

fig.tight_layout()
plt.show()
